# Stage 30 — Copula fitting (joint defaults)

**Owner:** Data Scientist · copula  
**Pipeline stage:** `pipelines.stage_30_copula`

Fit the joint-default copula from the marginal PDs and the correlation matrix, then persist the joint-default probability matrix and parameters. Clayton (lower-tail clustering) is the default; try `student_t` for symmetric tail dependence.

**Reads:** `persons_scored`, `corr_matrix`  
**Writes:** `joint_default_matrix`, `copula_params`

> This notebook is *modular*: it only needs its upstream artifacts to exist in the
> shared store. You can re-run just this stage without touching the others.


## 1. Setup

In [ ]:
# --- setup: make the project root importable + open the shared artifact store ---
import sys, os
ROOT = os.path.abspath("..")          # notebooks/ lives one level below the project root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from pipelines import ArtifactStore
store = ArtifactStore("output/etl")   # the SAME store every stage reads/writes
print("artifact store:", store.root)
print("existing artifacts:", store.list())


## 2. Run this stage

In [ ]:
from pipelines.stage_30_copula import run as run_copula

result = run_copula(store, copula_type="clayton", n_simulations=500)
print(result.summary())


## 3. Inspect what this stage produced

In [ ]:
import numpy as np
J = store.read_array("joint_default_matrix")
off = J[~np.eye(len(J), dtype=bool)]
print("joint-default matrix shape:", J.shape, "| avg off-diag P(D_i∩D_j):", off.mean())
print("copula_params:", store.read_json("copula_params"))
